# 09 · ML · Segunda técnica: CatBoost × LightGBM

**Squad 2 · Negócio em Tempo Real | Dupla José Vieira & Tiago Ramos**
**Problema P3:** estimar o `valor_total` do pedido a partir do contexto da compra.

Aplica o **CatBoost Regressor** nos mesmos dados, cenários e corte treino/teste do LightGBM do notebook 08 e
identifica qual dos dois é melhor. A entrada não muda: as mesmas 12 colunas da tabela de features, com as 4
categóricas declaradas como `cat_features` (o CatBoost trata categoria por estatística do alvo, sem one-hot).

| Entrada | Saída |
|---|---|
| `squad2/grupo2/ml/features_valor_pedido` | comparação nesta tela (o modelo escolhido segue no notebook 10) |

In [ ]:
%pip install -q lightgbm==4.7.0 catboost==1.2.10

In [ ]:
%run ./00_setup_config

## Parâmetros

Os dois modelos treinam em `log1p(valor_total)` e voltam para reais com `expm1`, como no 08. O LightGBM usa os
mesmos hiperparâmetros do 08; o CatBoost usa profundidade 6 e taxa 0,05, perto do padrão da biblioteca. Nenhum
dos dois é ajustado pelo teste.

In [ ]:
import lightgbm as lgb, numpy as np, pandas as pd, matplotlib.pyplot as plt, time
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, r2_score

CAMINHO_FEATURES = f"{BASE}/ml/features_valor_pedido"
CATEGORICAS = ["categoria_dominante_idx", "metodo_pagamento_idx", "periodo_idx", "uf_idx"]
FEATURES = ["hora", "dia_semana", "fim_de_semana", "n_itens", "qtd_total", "n_categorias", "qtd_por_item", "uf_informada"] + CATEGORICAS
PARAMS_LGB = dict(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=20, subsample=0.8, subsample_freq=1,
                  colsample_bytree=0.9, random_state=42, verbose=-1)
PARAMS_CB = dict(iterations=800, learning_rate=0.05, depth=6, loss_function="RMSE", random_seed=42, verbose=0, allow_writing_files=False)
AZUL, AZUL_CLARO, CINZA = "#1F4E8C", "#5B8FC9", "#8C8C8C"
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.titleweight": "bold", "axes.edgecolor": "#595959", "axes.grid": True, "grid.alpha": .3})

In [ ]:
fs = ler_delta(CAMINHO_FEATURES)
if fs is None: raise FileNotFoundError("Rode o notebook 07 antes: tabela de features nao encontrada")
df = fs.select("fonte", "conjunto", "id_pedido", "categoria_dominante", "flag_item_fora_curva", "valor_total", *FEATURES).toPandas()
df[CATEGORICAS] = df[CATEGORICAS].astype("int32")
df["eh_stream"] = (df.fonte == "stream").astype("int32")
print(df.groupby(["fonte", "conjunto"]).size().to_string())

## Treino dos dois modelos

Os 4 cenários do 08 (a histórico→histórico, b histórico→stream, c stream→stream, d stream sem item fora da curva),
com o mesmo baseline (média do ticket por categoria dominante no treino), e um cenário **e**: treino histórico +
stream sem os pedidos com item fora da curva, com a coluna `eh_stream`, testado no stream (o baseline do e é a média do treino do stream, a mesma do c). O e é a configuração
que o notebook 10 leva para produção, então é nele que a escolha vale.

**Regra de decisão:** ganha o menor MAE no cenário e. Se o bootstrap abaixo apontar empate, ganha o que treina
mais rápido. O tempo de treino é medido por isso. Os cenários a e b têm o mesmo treino, então o modelo do a é
reaproveitado no b.

In [ ]:
def parte(fonte, conjunto, sem_fora=False):
    d = df[(df.fonte == fonte) & (df.conjunto == conjunto)]
    return d[d.flag_item_fora_curva == 0] if sem_fora else d

def treinar(tipo, tr, feats):
    if tipo == "LightGBM": return lgb.LGBMRegressor(**PARAMS_LGB).fit(tr[feats], np.log1p(tr.valor_total), categorical_feature=CATEGORICAS)
    return CatBoostRegressor(**PARAMS_CB, cat_features=CATEGORICAS).fit(tr[feats], np.log1p(tr.valor_total))

treino_e = pd.concat([parte("historico", "treino"), parte("stream", "treino")]).query("flag_item_fora_curva == 0")

CENARIOS = {"a hist→hist": (parte("historico", "treino"), parte("historico", "teste")),
            "b hist→stream": (parte("historico", "treino"), parte("stream", "teste")),
            "c stream→stream": (parte("stream", "treino"), parte("stream", "teste")),
            "d stream sem fora da curva": (parte("stream", "treino", True), parte("stream", "teste", True)),
            "e hist+stream sem fora da curva→stream": (treino_e, parte("stream", "teste"))}
prev, linhas, modelos = {}, [], {}
for nome, (tr, te) in CENARIOS.items():
    feats = FEATURES + ["eh_stream"] if nome.startswith("e") else FEATURES
    tb = parte("stream", "treino") if nome.startswith("e") else tr
    base = te.categoria_dominante.map(tb.groupby("categoria_dominante").valor_total.mean()).fillna(tb.valor_total.mean())
    for tipo in ["LightGBM", "CatBoost"]:
        t0 = time.time()
        m = modelos[(nome, tipo)] = modelos[("a hist→hist", tipo)] if nome.startswith("b") else treinar(tipo, tr, feats)
        seg = time.time() - t0 if not nome.startswith("b") else np.nan
        p = prev[(nome, tipo)] = np.expm1(m.predict(te[feats]))
        linhas.append(dict(cenario=nome, modelo=tipo, n_treino=len(tr), n_teste=len(te), mae=mean_absolute_error(te.valor_total, p),
                           r2=r2_score(te.valor_total, p), mae_baseline=mean_absolute_error(te.valor_total, base), seg_treino=seg))
resultado = pd.DataFrame(linhas).round(3)
display(resultado)

## A diferença é real ou é ruído?

O teste do stream tem só 367 pedidos, então alguns reais de diferença no MAE podem ser acaso. Bootstrap pareado:
sorteia 2.000 vezes os pedidos do teste com reposição, mede `MAE CatBoost − MAE LightGBM` na mesma amostra e
guarda o intervalo de 95%. Se o intervalo cruza o zero, os dois modelos empatam naquele cenário.

In [ ]:
rng = np.random.default_rng(42)
def bootstrap_dif(y, p1, p2, n=2000, lote=200):
    d = np.abs(np.asarray(p2) - np.asarray(y)) - np.abs(np.asarray(p1) - np.asarray(y))
    medias = np.concatenate([d[rng.integers(0, len(d), (lote, len(d)))].mean(1) for _ in range(n // lote)])
    return np.percentile(medias, [2.5, 97.5])

linhas = []
for nome, (tr, te) in CENARIOS.items():
    lo, hi = bootstrap_dif(te.valor_total, prev[(nome, "LightGBM")], prev[(nome, "CatBoost")])
    dif = mean_absolute_error(te.valor_total, prev[(nome, "CatBoost")]) - mean_absolute_error(te.valor_total, prev[(nome, "LightGBM")])
    linhas.append(dict(cenario=nome, dif_mae_cat_menos_lgb=dif, ic95_inf=lo, ic95_sup=hi,
                       veredito="empate" if lo < 0 < hi else ("CatBoost melhor" if hi < 0 else "LightGBM melhor")))
comparacao = pd.DataFrame(linhas).round(2)
display(comparacao)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
piv = {m: resultado.pivot(index="cenario", columns="modelo", values=m).loc[list(CENARIOS)] for m in ["mae", "r2", "seg_treino"]}
base = resultado.drop_duplicates("cenario").set_index("cenario").loc[list(CENARIOS), "mae_baseline"]
x, w, rot = np.arange(len(CENARIOS)), .27, [c.split(" ")[0] for c in CENARIOS]
ax[0].bar(x - w, piv["mae"].LightGBM, w, color=AZUL, label="LightGBM"); ax[0].bar(x, piv["mae"].CatBoost, w, color=AZUL_CLARO, label="CatBoost")
ax[0].bar(x + w, base, w, color=CINZA, label="Baseline (média por categoria)"); ax[0].set_title("MAE (R$), menor é melhor")
ax[1].bar(x - w / 2, piv["r2"].LightGBM, w, color=AZUL, label="LightGBM"); ax[1].bar(x + w / 2, piv["r2"].CatBoost, w, color=AZUL_CLARO, label="CatBoost")
ax[1].set_title("R², maior é melhor")
ax[2].bar(x - w / 2, piv["seg_treino"].LightGBM, w, color=AZUL, label="LightGBM"); ax[2].bar(x + w / 2, piv["seg_treino"].CatBoost, w, color=AZUL_CLARO, label="CatBoost")
ax[2].set_title("Tempo de treino (s)")
for a in ax: a.set_xticks(x, rot); a.legend(fontsize=8, loc="upper left")
plt.suptitle("LightGBM × CatBoost nos cenários a a e (no b o modelo é o mesmo do a, sem novo treino)", fontweight="bold"); plt.tight_layout(); plt.show()

## Os dois modelos olham para as mesmas variáveis?

Importância normalizada em % (ganho no LightGBM, `PredictionValuesChange` no CatBoost) do modelo do cenário a.
As duas medidas não são a mesma conta, então a comparação vale pela **ordem** das variáveis, não pelo valor exato.
Se as duas técnicas concordam no que pesa, a leitura de negócio do 08 vale para qualquer uma.

In [ ]:
imp = pd.DataFrame({"LightGBM": modelos[("a hist→hist", "LightGBM")].booster_.feature_importance("gain"),
                    "CatBoost": modelos[("a hist→hist", "CatBoost")].get_feature_importance()}, index=FEATURES)
imp = (imp / imp.sum() * 100).sort_values("LightGBM")
fig, ax = plt.subplots(figsize=(8, 5)); y = np.arange(len(imp))
ax.barh(y + .2, imp.LightGBM, .4, color=AZUL, label="LightGBM"); ax.barh(y - .2, imp.CatBoost, .4, color=AZUL_CLARO, label="CatBoost")
ax.set_yticks(y, imp.index); ax.set_xlabel("% da importância total"); ax.set_title("Importância das variáveis, cenário a"); ax.legend()
plt.tight_layout(); plt.show()

## Resumo

Números da execução de teste (histórico 90.335 pedidos, tempo real 1.834; corte temporal 80/20):

| Cenário | MAE LightGBM | MAE CatBoost | R² LightGBM | R² CatBoost | Bootstrap (CatBoost − LightGBM) |
|---|---|---|---|---|---|
| a histórico → histórico | R$ 122,4 | R$ 122,4 | 0,82 | 0,82 | empate |
| b histórico → stream | R$ 149,4 | R$ 150,9 | 0,08 | 0,08 | empate |
| c stream → stream | R$ 137,0 | R$ 127,1 | 0,04 | 0,06 | CatBoost melhor (−R$ 9,9; IC −15,1 a −4,2) |
| d stream sem fora da curva | R$ 84,6 | R$ 81,2 | 0,44 | 0,45 | empate |
| e histórico + stream sem fora da curva → stream | R$ 122,8 | R$ 122,5 | 0,06 | 0,05 | empate (−R$ 0,3; IC −3,2 a 2,7) |

- **LightGBM e CatBoost empatam no erro; pela regra de decisão, fica o LightGBM.** No cenário e, que é o que vai
  para produção, a diferença é de R$ 0,30 no MAE, com intervalo cruzando o zero, e o LightGBM treina mais de 10
  vezes mais rápido (menos de 1 s contra 10 a 13 s nas execuções de teste). Também empatam no histórico (a), onde o
  volume é grande.
- **O CatBoost só ganha quando há pouco dado.** No cenário c (1.467 pedidos, só stream) ele erra R$ 9,90 a menos,
  e a diferença não é acaso; no d ele também erra menos, mas o bootstrap dá empate. Uma explicação provável é que
  o CatBoost regulariza melhor com amostra pequena (estatística do alvo por categoria e árvores simétricas); não
  testamos qual dos dois mecanismos responde pelo ganho. Quando o histórico entra no treino, a vantagem some.
- **As duas técnicas leem o problema do mesmo jeito.** A ordem das variáveis é a mesma: `qtd_total`, categoria
  dominante e nº de itens na frente; hora, pagamento e UF quase sem peso. O CatBoost dá mais peso à categoria,
  mas a conclusão de negócio do 08 vale para os dois.
- **A entrada não mudou.** As mesmas 12 colunas, com os índices das categóricas declarados como `cat_features`.
  O cenário e só acrescenta `eh_stream`, igual para as duas técnicas.

O notebook 10 segue com o LightGBM.